# 02 — Clinical score audit

**Purpose.** Validate and describe the existing score columns in the processed admission-level dataset. This notebook does not retrain models and does not overwrite `config['feature_sets']`.

An existing score column is treated as a supplied score value. It is not recalculated here from component variables. Any formula-specific recalculation should be implemented as reusable package code and compared with the supplied column.


In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
from pathlib import Path
import yaml
from icare_risk.data.audit import load_admission_file
from icare_risk.data.clinical_scores import score_summary, validate_score_columns

from icare_risk.utils.paths import ANALYSIS_YAML_PATH
with ANALYSIS_YAML_PATH.open() as f:
    config = yaml.safe_load(f)

dataset_path = config['outputs']['processed_dataset']
df = load_admission_file(dataset_path)
score_definitions = config['scores']
score_columns = {
    name: definition['source_column']
    for name, definition in score_definitions.items()
}

print(f"Processed dataset: {dataset_path}")
print(f"Admissions: {len(df):,}")
print('Configured scores:')
for name, column in score_columns.items():
    print(f"  {name}: source column = {column}")

Processed dataset: /app/data/synthetic/10000p/processed/admissions_with_split.parquet
Admissions: 10,000
Configured scores:
  charlson_score: source column = charlson_score
  kim_score: source column = kim_score
  tumbarello_score: source column = tumbarello_score
  jones_score: source column = jones_score
  gavaghan_score: source column = gavaghan_score
  holmgren_score: source column = holmgren_score


In [17]:
validation = validate_score_columns(
    df=df, score_columns=list(score_columns.values()),
)
display(validation)
missing_scores = validation.loc[~validation['exists'], 'score'].tolist()
if missing_scores:
    raise KeyError(f"Configured score columns are missing: {missing_scores}")

,score,exists,n_missing,n_unique,min,max
0,charlson_score,True,0,13,0.0,15.0
1,kim_score,True,0,4,0.0,6.0
2,tumbarello_score,True,0,5,0.0,6.0
3,jones_score,True,0,2,0.0,2.0
4,gavaghan_score,True,0,2,0.0,1.0
5,holmgren_score,True,0,1,0.0,0.0


In [18]:
summary = score_summary(
    df=df,
    score_columns=list(score_columns.values()),
)
display(summary)

# Check score distributions by temporal split when the split label exists.
if 'split' in df.columns:
    print('Score distributions by split')
    for split_name, split_df in df.groupby('split', dropna=False):
        print(f"\n--- {split_name} ---")
        display(score_summary(split_df, list(score_columns.values())))

print('No configuration values were overwritten. Feature sets remain those defined in analysis.yaml.')

,score,n,n_missing,mean,std,median,min,max
0,charlson_score,10000,0,2.2851,1.733355,2.0,0.0,15.0
1,kim_score,10000,0,0.1206,0.372920,0.0,0.0,6.0
2,tumbarello_score,10000,0,0.2309,0.649790,0.0,0.0,6.0
3,jones_score,10000,0,0.0786,0.388635,0.0,0.0,2.0
4,gavaghan_score,10000,0,0.0007,0.026450,0.0,0.0,1.0
5,holmgren_score,10000,0,0.0000,0.000000,0.0,0.0,0.0


Score distributions by split

--- test ---


,score,n,n_missing,mean,std,median,min,max
0,charlson_score,2538,0,2.622143,1.750796,2.0,0.0,15.0
1,kim_score,2538,0,0.113475,0.329426,0.0,0.0,5.0
2,tumbarello_score,2538,0,0.222616,0.630090,0.0,0.0,4.0
3,jones_score,2538,0,0.078014,0.387300,0.0,0.0,2.0
4,gavaghan_score,2538,0,0.001970,0.044350,0.0,0.0,1.0
5,holmgren_score,2538,0,0.000000,0.000000,0.0,0.0,0.0



--- train ---


,score,n,n_missing,mean,std,median,min,max
0,charlson_score,5958,0,2.111615,1.702223,2.0,0.0,11.0
1,kim_score,5958,0,0.118328,0.348039,0.0,0.0,5.0
2,tumbarello_score,5958,0,0.231286,0.644613,0.0,0.0,4.0
3,jones_score,5958,0,0.078550,0.388529,0.0,0.0,2.0
4,gavaghan_score,5958,0,0.000336,0.018320,0.0,0.0,1.0
5,holmgren_score,5958,0,0.000000,0.000000,0.0,0.0,0.0



--- validation ---


,score,n,n_missing,mean,std,median,min,max
0,charlson_score,1504,0,2.403590,1.733399,2.0,0.0,11.0
1,kim_score,1504,0,0.141622,0.511241,0.0,0.0,6.0
2,tumbarello_score,1504,0,0.243351,0.701170,0.0,0.0,6.0
3,jones_score,1504,0,0.079787,0.391549,0.0,0.0,2.0
4,gavaghan_score,1504,0,0.000000,0.000000,0.0,0.0,0.0
5,holmgren_score,1504,0,0.000000,0.000000,0.0,0.0,0.0


No configuration values were overwritten. Feature sets remain those defined in analysis.yaml.


In [19]:
# Keep a stable, explicit list for later model feature-set definitions.
config['feature_sets'] = {
    name: {'columns': [definition['source_column']]}
    for name, definition in score_definitions.items()
}
print(config['feature_sets'])

{'charlson_score': {'columns': ['charlson_score']}, 'kim_score': {'columns': ['kim_score']}, 'tumbarello_score': {'columns': ['tumbarello_score']}, 'jones_score': {'columns': ['jones_score']}, 'gavaghan_score': {'columns': ['gavaghan_score']}, 'holmgren_score': {'columns': ['holmgren_score']}}
